In [14]:
from pathlib import Path
import duckdb

DB = Path("../data/processed/calgary_snow.duckdb")
SQL_DIR = Path("../sql")

def q(query):
    """Run a SQL query and return a DataFrame. Closes the file right after."""
    with duckdb.connect(str(DB), read_only=True) as con:
        return con.execute(query).df()

def run_sql_file(name):
    """Run a file from the sql/ folder, e.g. run_sql_file('snow_events')."""
    return q((SQL_DIR / f"{name}.sql").read_text())

In [15]:
q("SHOW TABLES")

,name
0,incidents
1,weather


In [16]:
q("""
SELECT *
FROM incidents
LIMIT 20
""")

,id,incident_info,description,quadrant,latitude,longitude,start_dt,date,hour
0,2023-03-21 18:02:00+00:00-114.059945250.97957572,Heritage Drive and Fairmount Drive SE,Traffic incident.,SE,50.979576,-114.059945,2023-03-21 12:02:00,2023-03-21,12
1,2026-10-01 06:10:22+00:0050.949792034615726-11...,Anderson Road and Bonaventure Drive SE,Multi-vehicle incident. Blocking three WB lanes.,SE,50.949792,-114.062054,2026-10-01 00:10:22,2026-10-01,0
2,2026-10-01 07:00:42+00:0051.04790098945052-114...,6 Street and 7 Street SW,There is an incident involving a pedestrian- E...,SW,51.047901,-114.078681,2026-10-01 01:00:42,2026-10-01,1
3,2026-10-01 12:54:37+00:0051.00468640241671-113...,Southbound 52 Street and 54 Avenue SE,Multi-vehicle incident. Blocking multiple lanes,SE,51.004686,-113.958628,2026-10-01 06:54:37,2026-10-01,6
4,2026-10-01 13:26:30+00:0050.95099908888404-113...,Northbound 52 Street and 114 Avenue SE,Multi-vehicle incident. Blocking the NB right ...,SE,50.950999,-113.958114,2026-10-01 07:26:30,2026-10-01,7
5,2026-10-01 13:41:47+00:0051.095988596187176-11...,Eastbound McKnight Boulevard after 52 Street NE,Stalled vehicle. Blocking the EB right lane.,NE,51.095989,-113.954577,2026-10-01 07:41:47,2026-10-01,7
6,2026-10-01 13:56:04+00:0051.008757529521205-11...,Glenmore Trail and Stoney Trail SW,Traffic incident.,SW,51.008758,-114.152321,2026-10-01 07:56:04,2026-10-01,7
7,2026-10-01 14:37:38+00:0051.14179506750416-114...,Airport Trail and Deerfoot Trail NE,Multi-vehicle incident. Blocking multiple lanes,NE,51.141795,-114.038265,2026-10-01 08:37:38,2026-10-01,8
8,2026-10-01 14:40:44+00:0051.05238717718435-114...,Eastbound Bow Trail and Sarcee Trail SW,Two vehicle incident. Partially blocking the E...,SW,51.052387,-114.164082,2026-10-01 08:40:44,2026-10-01,8
9,2026-10-01 15:22:04+00:0051.04289700684066-114...,8 Street and 11 Avenue SW,Traffic signals work in progress. CPS directin...,SW,51.042897,-114.081475,2026-10-01 09:22:04,2026-10-01,9


In [17]:
q("""
SELECT *
FROM weather
LIMIT 20
""")

,date,max_temp,min_temp,mean_temp,total_snow,total_precip,snow_on_ground
0,2017-01-01,-7.8,-17.4,-12.6,3.6,2.0,12.0
1,2017-01-02,-16.1,-26.1,-21.1,0.0,0.0,14.0
2,2017-01-03,-15.5,-22.4,-19.0,0.4,0.3,14.0
3,2017-01-04,-7.2,-17.2,-12.2,0.0,0.0,13.0
4,2017-01-05,-3.5,-15.8,-9.7,2.2,1.2,13.0
5,2017-01-06,-9.9,-17.0,-13.5,3.0,1.6,15.0
6,2017-01-07,-14.9,-19.2,-17.1,0.4,0.4,16.0
7,2017-01-08,-17.4,-19.8,-18.6,0.0,0.0,16.0
8,2017-01-09,-19.5,-23.1,-21.3,4.3,4.2,20.0
9,2017-01-10,-16.2,-22.1,-19.2,0.4,0.4,20.0


In [19]:
q("""SELECT date, COUNT(*) AS incidents
FROM incidents
GROUP BY date""")

q("""SELECT MIN(date) AS first_day, 
MAX(date) AS last_day 
FROM incidents""")

q("""
WITH daily_incidents AS (
    SELECT date, COUNT(*) AS incidents
    FROM incidents
    GROUP BY date
),
date_range AS (
    SELECT MIN(date) AS first_day, MAX(date) AS last_day FROM incidents
)
SELECT
    w.date,
    w.total_snow,
    w.snow_on_ground,
    w.mean_temp,
    CASE
        WHEN COALESCE(w.total_snow, 0) >= 5 THEN 'heavy_snow'
        WHEN COALESCE(w.total_snow, 0) > 0 THEN 'light_snow'
        WHEN COALESCE(w.snow_on_ground, 0) >= 1 THEN 'snow_on_ground'
        ELSE 'clear'
    END AS snow_category,
    COALESCE(d.incidents, 0) AS incidents
FROM weather w
LEFT JOIN daily_incidents d ON d.date = w.date
CROSS JOIN date_range r
WHERE w.date BETWEEN r.first_day AND r.last_day
ORDER BY w.date
""")

,date,total_snow,snow_on_ground,mean_temp,snow_category,incidents
0,2017-01-01,3.6,12.0,-12.6,light_snow,0
1,2017-01-02,0.0,14.0,-21.1,snow_on_ground,0
2,2017-01-03,0.4,14.0,-19.0,light_snow,0
3,2017-01-04,0.0,13.0,-12.2,snow_on_ground,0
4,2017-01-05,2.2,13.0,-9.7,light_snow,0
...,...,...,...,...,...,...
2049,2026-10-23,2.0,0.0,3.3,light_snow,31
2050,2026-10-24,1.0,1.0,0.2,light_snow,23
2051,2026-10-25,1.0,1.0,-0.9,light_snow,23
2052,2026-10-26,0.0,NaN,-0.7,clear,24


In [21]:
q("""WITH daily_incidents AS (
    SELECT date, COUNT(*) AS incidents
    FROM incidents
    GROUP BY date
),
date_range AS (
    SELECT MIN(date) AS first_day, MAX(date) AS last_day FROM incidents
),
days AS (
    SELECT
        CASE
            WHEN COALESCE(w.total_snow, 0) >= 5 THEN 'heavy_snow'
            WHEN COALESCE(w.total_snow, 0) > 0 THEN 'light_snow'
            WHEN COALESCE(w.snow_on_ground, 0) >= 1 THEN 'snow_on_ground'
            ELSE 'clear'
        END AS snow_category,
        COALESCE(d.incidents, 0) AS incidents
    FROM weather w
    LEFT JOIN daily_incidents d ON d.date = w.date
    CROSS JOIN date_range r
    WHERE w.date BETWEEN r.first_day AND r.last_day
),
summary AS (
    SELECT
        snow_category,
        COUNT(*)                     AS num_days,
        SUM(incidents)               AS total_incidents,
        ROUND(AVG(incidents * 1.0), 2) AS avg_incidents_per_day
    FROM days
    GROUP BY snow_category
)
SELECT
    s.snow_category,
    s.num_days,
    s.total_incidents,
    s.avg_incidents_per_day,
    ROUND(s.avg_incidents_per_day / c.avg_incidents_per_day, 2) AS ratio_vs_clear
FROM summary s
CROSS JOIN (SELECT avg_incidents_per_day FROM summary WHERE snow_category = 'clear') c
ORDER BY s.avg_incidents_per_day DESC;
""")

,snow_category,num_days,total_incidents,avg_incidents_per_day,ratio_vs_clear
0,heavy_snow,87,2486.0,28.57,2.04
1,light_snow,412,7154.0,17.36,1.24
2,snow_on_ground,735,10382.0,14.13,1.01
3,clear,820,11483.0,14.00,1.00


In [23]:
q("""
WITH date_range AS (
    SELECT MIN(date) AS first_day, MAX(date) AS last_day FROM incidents
),
day_type AS (
    SELECT
        w.date,
        CASE WHEN COALESCE(w.total_snow, 0) > 0 THEN 'snow_day' ELSE 'no_snow' END AS day_type
    FROM weather w
    CROSS JOIN date_range r
    WHERE w.date BETWEEN r.first_day AND r.last_day
),
day_counts AS (
    SELECT day_type, COUNT(*) AS num_days FROM day_type GROUP BY day_type
)
SELECT
    i.quadrant,
    t.day_type,
    COUNT(*) AS total_incidents,
    ROUND(COUNT(*) * 1.0 / dc.num_days, 2) AS avg_incidents_per_day
FROM incidents i
JOIN day_type t  ON t.date = i.date
JOIN day_counts dc ON dc.day_type = t.day_type
WHERE i.quadrant IS NOT NULL
GROUP BY i.quadrant, t.day_type, dc.num_days
ORDER BY i.quadrant, t.day_type
""")

,quadrant,day_type,total_incidents,avg_incidents_per_day
0,NE,no_snow,5927,3.81
1,NE,snow_day,2565,5.14
2,NW,no_snow,4833,3.11
3,NW,snow_day,2260,4.53
4,SE,no_snow,6784,4.36
5,SE,snow_day,3068,6.15
6,SW,no_snow,4321,2.78
7,SW,snow_day,1747,3.50


In [24]:
q(""" 
WITH date_range AS (
    SELECT MIN(date) AS first_day, MAX(date) AS last_day FROM incidents
),
day_type AS (
    SELECT
        w.date,
        CASE WHEN COALESCE(w.total_snow, 0) > 0 THEN 'snow_day' ELSE 'no_snow' END AS day_type
    FROM weather w
    CROSS JOIN date_range r
    WHERE w.date BETWEEN r.first_day AND r.last_day
),
day_counts AS (
    SELECT day_type, COUNT(*) AS num_days FROM day_type GROUP BY day_type
)
SELECT
    i.hour,
    t.day_type,
    ROUND(COUNT(*) * 1.0 / dc.num_days, 2) AS avg_incidents_per_day
FROM incidents i
JOIN day_type t  ON t.date = i.date
JOIN day_counts dc ON dc.day_type = t.day_type
GROUP BY i.hour, t.day_type, dc.num_days
ORDER BY i.hour, t.day_type
""")

,hour,day_type,avg_incidents_per_day
0,0,no_snow,0.19
1,0,snow_day,0.27
2,1,no_snow,0.19
3,1,snow_day,0.21
4,2,no_snow,0.14
5,2,snow_day,0.15
6,3,no_snow,0.10
7,3,snow_day,0.14
8,4,no_snow,0.13
9,4,snow_day,0.16
